In [1]:
import pandas as pd
import re
from ollama import chat, ChatResponse
from tqdm import tqdm  # 用於顯示進度條

In [2]:
# --- 設定 ---
FILE_PATH = './test/RACE_High_Combined.xlsx'     # 資料集路徑

In [3]:
# --- 函數：呼叫 Ollama (使用官方庫) ---
def query_ollama(prompt, model='gemma3:12b', context_len=1536, think=False):
    try:
        # 使用 ollama.chat 直接呼叫
        response: ChatResponse = chat(
            model=model,
            messages=[
                {'role': 'user', 'content': prompt}
            ],
            # 您可以在此加入 options，例如 temperature
            options={
                'num_ctx': context_len,  # 在這裡指定 Context Length
                'temperature': 0.0  # 讓輸出結果穩定
            },
            think=think  # 設定為 True 以獲取推理欄位
        )
        # 回傳模型的內容
        return response.message.content
    except Exception as e:
        print(f"Error calling Ollama: {e}")
        return ""

In [4]:
# --- 函數：解析模型輸出 ---
def extract_answer(text):
    """
    從模型的回答中提取選項數字 (1, 2, 3, 4)。
    DeepSeek R1 會輸出 <think>...</think> 思考過程，需要過濾。
    """
    if not text:
        return None

    # 1. 移除 <think> 標籤內的內容 (DeepSeek R1 特有)
    # flags=re.DOTALL 確保 . 可以匹配換行符號
    text_cleaned = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()
    
    # 2. 策略 A: 尋找明確的 "Answer: X" 格式 (Prompt 中有要求)
    match = re.search(r'Answer:\s*([1-4])', text_cleaned, re.IGNORECASE)
    if match:
        return int(match.group(1))
    
    # 3. 策略 B: 如果沒有標準格式，尋找最後出現的數字 1-4
    # 這是一個備用方案
    numbers = re.findall(r'\b([1-4])\b', text_cleaned)
    if numbers:
        return int(numbers[-1])
        
    return None  # 解析失敗

In [5]:
# --- 主程式 ---
# 1. 讀取資料
print(f"正在讀取 {FILE_PATH}...")
try:
    df = pd.read_excel(FILE_PATH)
except Exception as e:
    print(f"讀取檔案失敗: {e}")

# 檢查必要的欄位
required_columns = ['ID', 'Article', 'Question', 'Option1', 'Option2', 'Option3', 'Option4', 'Answer']
if not all(col in df.columns for col in required_columns):
    print(f"錯誤: Excel 檔案缺少必要的欄位。需要: {required_columns}")

正在讀取 ./test/RACE_High_Combined.xlsx...


In [6]:
df

,ID,Original_ID,Article,Question,Option1,Option2,Option3,Option4,Answer
0,1,high10001.txt,Studies show that you may be lied to every day...,From Para.1 we learn that lying is very _ .,harmful,easy,interesting,common,4
1,2,high10001.txt,Studies show that you may be lied to every day...,"According to the passage, a lie works when _ .",the liar's words are sweet enough,it is given to a complete stranger,someone gives the liar cooperation,we are willing to lie for the dignity,3
2,3,high10001.txt,Studies show that you may be lied to every day...,Lying is complex because _ .,it is practiced by clever ones,people are for it as well as against it,ancient stories are full of lies,it is the whole part of great culture,2
3,4,high10001.txt,Studies show that you may be lied to every day...,The examples of kids lying in the passage show...,lying is a sign of intelligent development,lying is good for children's growing up,at what age children begin to tell lies,9-year-old children are masters of lying,1
4,5,high10001.txt,Studies show that you may be lied to every day...,What will the writer most likely talk about if...,How to become clever liars.,How to avoid stupid lies.,How to get through the bushes.,How to tell truth from lies.,4
...,...,...,...,...,...,...,...,...,...
3493,3494,high996.txt,Onceupon a time there was a wise man that used...,"One day, the wise man saw a young man _ .",dancing along the beach,walking with a dancer,picking up starfish for sale,trying to save as many starfishes as possible,4
3494,3495,high996.txt,Onceupon a time there was a wise man that used...,The writer told this story to show us _ .,how and where we can write a good article,everyone can do something for the future,wise men are sometimes stupid,young men are in fact wiser than old people,2
3495,3496,high9985.txt,British MPs have voted in favour of the crea...,"According to the passage, which of the followi...",The new technique is intended to provide the p...,The mitochondrial disease can lead to various ...,Britain is the first country to have created t...,All of Ms Bernadi's children died of a genetic...,3
3496,3497,high9985.txt,British MPs have voted in favour of the crea...,It can be inferred from the passage that _ .,Mitochondria are inside nearly every cell that...,DNA of Mitochondria can affect a person's appe...,in medical science ethical factors should be t...,the donor woman's DNA will not be found in the...,3


In [7]:
# --- 主程式 ---
def main():
    correct_count = 0
    total_questions = len(df)
    results = []

    print(f"開始使用 {MODEL_NAME} 處理 {total_questions} 個問題...")

    # 2. 遍歷每一列進行預測
    for index, row in tqdm(df.iterrows(), total=total_questions):
        article = row['Article']
        question = row['Question']
        # 建立選項字典方便取用
        options = {
            1: row['Option1'],
            2: row['Option2'],
            3: row['Option3'],
            4: row['Option4']
        }
        actual_answer = row['Answer']
        #print(actual_answer)

        # 1. 過濾掉 NaN 或空值的選項 (假設 options 是 dict 或 list)
        # 這裡用 str(v) != 'nan' 是為了處理 pandas 的情況
        valid_options = {k: v for k, v in options.items() if v and str(v).lower() != 'nan'}
        
        # 2. 動態生成選項字串
        options_text = "\n".join([f"{k}. {v}" for k, v in valid_options.items()])
        valid_numbers = ", ".join([str(k) for k in valid_options.keys()])

        # 建構 Prompt
        # 這裡的設計是為了讓 DeepSeek R1 能夠先思考再回答，並最後輸出固定格式
        prompt = f"""
Read the following article and answer the multiple-choice question.

Article:
{article}

Question:
{question}

Options:
{options_text}

Instruction:
Analyze the article and select the best option from the available choices listed above ({valid_numbers}). 
If an option is missing or empty, do not select it.

Finally, output the answer in the exact format: "Answer: X", where X is the option number.
Do not output anything else after the answer.
"""
        
        # 呼叫模型
        response_text = query_ollama(prompt, model=MODEL_NAME, think=THINK_MODE)
        
        # 解析答案
        predicted_answer = extract_answer(response_text)
        
        # 比對結果
        is_correct = False
        try:
            # 處理 Excel 讀入的答案可能是浮點數或字串的情況
            actual_answer_int = int(actual_answer)
            if predicted_answer is not None and predicted_answer == actual_answer_int:
                correct_count += 1
                is_correct = True
            #else:
                #print("Article:", article)
                #print("Question:", question)
                #print("Option:", "1.", row['Option1'], "2.", row['Option2'], "3.", row['Option3'], "4.", row['Option4'])
                #print("Answer:", actual_answer)
                #print("Predicted Answer:", predicted_answer)
                #print('-'*30)
        except ValueError:
            pass # 忽略無效的標準答案格式

        # 儲存詳細結果
        results.append({
            "ID": row['ID'],
            "Actual": actual_answer,
            "Predicted": predicted_answer,
            "Correct": is_correct,
            "Model_Response": response_text
        })

        #print(predicted_answer)
        #if (index+1) % 500 == 0:
            #print(correct_count, " / ", (index+1), " = ", 100.0*correct_count/(index+1), "%")
            

    # 3. 計算並輸出正確率
    accuracy = (correct_count / total_questions) * 100 if total_questions > 0 else 0
    print("\n" + "="*30)
    print(f"處理完成！")
    print(f"總題數: {total_questions}")
    print(f"答對題數: {correct_count}")
    print(f"正確率 (Accuracy): {accuracy:.2f}%")
    print("="*30)

    # 4. 將結果存檔
    output_file = "result_analysis.xlsx"
    pd.DataFrame(results).to_excel(output_file, index=False)
    print(f"詳細結果已儲存至 {output_file}")

In [8]:
MODEL_NAME = 'gemma3:12b'  # 模型名稱
THINK_MODE = False
main()

開始使用 gemma3:12b 處理 3498 個問題...


100%|██████████████████████████████████████████████████████████████████████████████| 3498/3498 [16:48<00:00,  3.47it/s]



處理完成！
總題數: 3498
答對題數: 2976
正確率 (Accuracy): 85.08%
詳細結果已儲存至 result_analysis.xlsx


In [9]:
MODEL_NAME = 'qwen3:14b'  # 模型名稱
THINK_MODE = False
#THINK_MODE = True
main()

開始使用 qwen3:14b 處理 3498 個問題...


100%|██████████████████████████████████████████████████████████████████████████████| 3498/3498 [13:28<00:00,  4.33it/s]



處理完成！
總題數: 3498
答對題數: 3111
正確率 (Accuracy): 88.94%
詳細結果已儲存至 result_analysis.xlsx


In [8]:
MODEL_NAME = 'deepseek-r1:14b'  # 模型名稱
THINK_MODE = False
#THINK_MODE = True
main()

開始使用 deepseek-r1:14b 處理 3498 個問題...


100%|████████████████████████████████████████████████████████████████████████████| 3498/3498 [2:05:07<00:00,  2.15s/it]



處理完成！
總題數: 3498
答對題數: 3021
正確率 (Accuracy): 86.36%
詳細結果已儲存至 result_analysis.xlsx


In [11]:
MODEL_NAME = 'phi3:14b'  # 模型名稱
THINK_MODE = False
#THINK_MODE = True
main()

開始使用 phi3:14b 處理 3498 個問題...


100%|██████████████████████████████████████████████████████████████████████████████| 3498/3498 [17:50<00:00,  3.27it/s]



處理完成！
總題數: 3498
答對題數: 2921
正確率 (Accuracy): 83.50%
詳細結果已儲存至 result_analysis.xlsx


In [9]:
MODEL_NAME = 'phi4:14b'  # 模型名稱
THINK_MODE = False
#THINK_MODE = True
main()

開始使用 phi4:14b 處理 3498 個問題...


100%|████████████████████████████████████████████████████████████████████████████| 3498/3498 [3:59:53<00:00,  4.11s/it]



處理完成！
總題數: 3498
答對題數: 3076
正確率 (Accuracy): 87.94%
詳細結果已儲存至 result_analysis.xlsx


In [ ]:
MODEL_NAME = 'phi4-reasoning:14b'  # 模型名稱
THINK_MODE = False
#THINK_MODE = True
main()